# LLM Fine-Tuning: Domain-Specific Q&A via QLoRA
### Fine-Tuning Mistral-7B / Llama-3-8B on Finance & Regulatory Compliance

This notebook demonstrates the end-to-end QLoRA fine-tuning pipeline following the implementation plan:
1. **Phase 1: Environment & Setup** — 4-bit quantization, dependencies, GPU verification.
2. **Phase 2: Data Preparation** — Instruction-tuning format (`instruction`, `input`, `output`).
3. **Phase 3: QLoRA Configuration** — BitsAndBytes NF4, LoRA rank 16 / alpha 32.
4. **Phase 4: SFTTrainer Execution** — Supervised fine-tuning with cosine LR scheduler.
5. **Phase 5: Evaluation & Inference** — Direct side-by-side comparison of Base Model vs Fine-Tuned Model.
6. **Phase 6: Adapter Export** — Saving PEFT checkpoint for FastAPI serving.

In [ ]:
# 1. GPU Check & Package Installation
!nvidia-smi
!pip install -q -U torch transformers peft bitsandbytes trl accelerate datasets scipy sentencepiece

In [ ]:
import os
import torch
import json
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### 2. Base Model & Quantization Configuration
Loading base 7B/8B checkpoint in 4-bit NormalFloat (NF4) with double quantization and bfloat16 compute.

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2" # Or "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
print("Base model successfully loaded in 4-bit precision!")

### 3. LoRA Configuration
Applying parameter-efficient Low-Rank Adaptation targeting all linear projection layers.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 4. Dataset Loading & Instruction Formatting

In [ ]:
# Load cleaned JSONL train / eval splits
with open("data/processed/train.jsonl", "r", encoding="utf-8") as f:
    train_records = [json.loads(line) for line in f]

with open("data/processed/eval.jsonl", "r", encoding="utf-8") as f:
    eval_records = [json.loads(line) for line in f]

def format_prompts(batch):
    formatted_texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        content = f"{inst}\n\nContext: {inp}" if inp else inst
        formatted_texts.append(f"<s>[INST] {content} [/INST] {out}</s>")
    return formatted_texts

train_dataset = Dataset.from_list(train_records)
eval_dataset = Dataset.from_list(eval_records)
print(f"Train samples: {len(train_dataset)}, Eval samples: {len(eval_dataset)}")

### 5. Training with TRL SFTTrainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./adapters/finops_qlora_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    formatting_func=format_prompts,
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained("./adapters/finops_qlora_adapter")
tokenizer.save_pretrained("./adapters/finops_qlora_adapter")
print("Training complete! Adapter saved to ./adapters/finops_qlora_adapter")

### 6. Side-by-Side Dual Inference Validation

In [ ]:
prompt = "How should our cloud infrastructure handle sudden spike traffic in an autoscaling Kubernetes cluster without exceeding our monthly reserved instance budget?"
formatted_input = f"<s>[INST] {prompt} [/INST]"

inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.7, top_p=0.95)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("=== Fine-Tuned Model Generation ===\n")
print(response)